In [ ]:
from pyspark.sql.functions import col, to_timestamp, current_timestamp

In [ ]:
dbutils.widgets.text("catalog", "olist_project_dev")

dbutils.widgets.text("bronze_schema", "olist_bronze")
dbutils.widgets.text("raw_olist_orders_table", "olist_orders")

dbutils.widgets.text("silver_schema", "olist_silver")
dbutils.widgets.text("orders_table", "orders_silver")

In [ ]:
catalog = dbutils.widgets.get("catalog")

bronze_schema = dbutils.widgets.get("bronze_schema")
raw_olist_orders_table_name = dbutils.widgets.get("raw_olist_orders_table")

silver_schema = dbutils.widgets.get("silver_schema")
orders_table_name = dbutils.widgets.get("orders_table")

In [ ]:
raw_olist_orders_df = spark.table(f"{catalog}.{bronze_schema}.{raw_olist_orders_table_name}")

In [ ]:
if not spark.catalog.tableExists(f"{catalog}.{silver_schema}.{orders_table_name}"):
    spark.sql(
        f"""
        CREATE TABLE {catalog}.{silver_schema}.{orders_table_name} (
            orderId STRING,
            customerId STRING,
            orderStatus STRING,
            orderPurchaseTimestamp TIMESTAMP,
            orderApprovedAt TIMESTAMP,
            orderDeliveredCarrierDate TIMESTAMP,
            orderDeliveredCustomerDate TIMESTAMP,
            orderEstimatedDeliveryDate TIMESTAMP,
            processedTimestamp TIMESTAMP
        )
        TBLPROPERTIES (
            'delta.autoOptimize.optimizeWrite' = 'true',
            'delta.autoOptimize.autoCompact' = 'true'
        )
        """
    )

In [ ]:
orders_silver_df = (
    raw_olist_orders_df
    .where(
        (col("order_id").rlike("^[0-9a-fA-F]{32}$")) & (col("customer_id").rlike("^[0-9a-fA-F]{32}$"))
    )
    .select(
        col("order_id").cast("string").alias("orderId"),
        col("customer_id").cast("string").alias("customerId"),
        col("order_status").cast("string").alias("orderStatus"),
        to_timestamp(col("order_purchase_timestamp"), "yyyy-MM-dd HH:mm:ss").alias("orderPurchaseTimestamp"),
        to_timestamp(col("order_approved_at"), "yyyy-MM-dd HH:mm:ss").alias("orderApprovedAt"),
        to_timestamp(col("order_delivered_carrier_date"), "yyyy-MM-dd HH:mm:ss").alias("orderDeliveredCarrierDate"),
        to_timestamp(col("order_delivered_customer_date"), "yyyy-MM-dd HH:mm:ss").alias("orderDeliveredCustomerDate"),
        to_timestamp(col("order_estimated_delivery_date"), "yyyy-MM-dd HH:mm:ss").alias("orderEstimatedDeliveryDate")
    )
    .withColumn("processedTimestamp", current_timestamp())
)

In [ ]:
spark.createTemporaryView("orders_silver_view", orders_silver_df)

spark.sql(f"""
    MERGE INTO {catalog}.{silver_schema}.{orders_table_name} AS target
    USING orders_silver_view AS source
    ON target.orderId = source.orderId
    WHEN MATCHED THEN
        UPDATE SET *
    WHEN NOT MATCHED THEN
        INSERT *
    """)